## 📋 **Scenario: Esplorazione Mineraria - Sondaggi e Analisi Geochimiche**

Un'azienda mineraria ha effettuato una campagna di sondaggi in diverse aree. Per ogni sondaggio sono stati raccolti campioni a diverse profondità, e sono stati analizzati in laboratorio.

## 🎯 **QUESITI (da risolvere con JOIN)**

### **A. Pulizia dati**
1. Crea due tabelle temporanee pulite con:
   - Formati uniformi (uppercase, trim)
   - Date in YYYY-MM-DD
   - Valori numerici (ND → NULL)

### **B. Analisi esplorativa con JOIN**

**1. Quali sondaggi hanno la media più alta di Au?**
- Mostra `Zona`, `ID_Sondaggio`, `media_Au_ppm`
- Ordina per media decrescente

**2. Quali zone hanno la migliore mineralizzazione?**
- Per ogni zona, calcola:
  - Media Au
  - Media Ag  
  - Media Cu
  - Numero campioni analizzati

**3. Qual è la correlazione tra profondità e tenore in Au?**
- Usa `(Da_m + A_m)/2` come profondità media del campione
- Calcola coefficiente di correlazione di Pearson

**4. Quali sondaggi hanno campioni con Au > 1 ppm?**
- Mostra `Zona`, `ID_Sondaggio`, profondità del campione, Au_ppm
- Ordina per Au decrescente

**5. Esiste una relazione tra attrezzatura utilizzata e tenori?**
- Per ogni tipo di attrezzatura, calcola la media Au nei campioni associati

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

sondaggi = {
    'ID_Sondaggio': ['SD-001', 'sd-001', 'SD002', 'SD-003'],
    'Data_Sondaggio': ['2024-01-10', '10/02/2024', '2024.03.15', '2024-04-01'],
    'Zona': ['Zona_A', 'zona a', 'Zona_B', 'Zona_C'],
    'Profondità_Totale_m': ['150', '200.5', '175', 'zero'],
    'Attrezzatura': ['Rotary', 'percussione', 'ROTARY', 'diamond']
}

analisi_campioni = {
    'ID_Campione': ['CAMP-101', 'camp-101', 'C102', 'C103', 'C104'],
    'ID_Sondaggio': ['SD-001', 'SD001', 'SD-002', 'SD-002', 'SD-001'],
    'Da_m': ['0.0', '10.5', '15', '20', '25'],
    'A_m': ['1.0', '11.5', '16', '25', '30'],
    'Au_ppm': ['0.5', '1.2', 'ND', '2.5', '0'],
    'Ag_ppm': ['2.1', '0.5', '1.5', 'ND', '0'],
    'Cu_ppm': ['120', '250', 'ND', '80', '0']
}

df_sondaggi = pd.DataFrame(sondaggi)
df_analisi_campioni = pd.DataFrame(analisi_campioni)

df_sondaggi.to_sql('sondaggi', conn, index=False, if_exists='replace')
df_analisi_campioni.to_sql('analisi_campioni', conn, index=False, if_exists='replace')

print("=== DATI GREZZI SONDAGGI ===")
print(df_sondaggi)

print("\n\n")

print("=== DATI GREZZI ANALISI CAMPIONI ===")
print(df_analisi_campioni)

"""Per verificare che SQLite abbia caricato davvero le tabelle, aggiungi anche:
print("\n=== TABELLE IN SQLITE ===")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))

print("\n=== CONTENUTO TABELLA SONDAGGI ===")
print(pd.read_sql("SELECT * FROM sondaggi;", conn))

print("\n=== CONTENUTO TABELLA ANALISI_CAMPIONI ===")
print(pd.read_sql("SELECT * FROM analisi_campioni;", conn))"""

=== DATI GREZZI SONDAGGI ===
  ID_Sondaggio Data_Sondaggio    Zona Profondità_Totale_m Attrezzatura
0       SD-001     2024-01-10  Zona_A                 150       Rotary
1       sd-001     10/02/2024  zona a               200.5  percussione
2        SD002     2024.03.15  Zona_B                 175       ROTARY
3       SD-003     2024-04-01  Zona_C                zero      diamond



=== DATI GREZZI ANALISI CAMPIONI ===
  ID_Campione ID_Sondaggio  Da_m   A_m Au_ppm Ag_ppm Cu_ppm
0    CAMP-101       SD-001   0.0   1.0    0.5    2.1    120
1    camp-101        SD001  10.5  11.5    1.2    0.5    250
2        C102       SD-002    15    16     ND    1.5     ND
3        C103       SD-002    20    25    2.5     ND     80
4        C104       SD-001    25    30      0      0      0


'Per verificare che SQLite abbia caricato davvero le tabelle, aggiungi anche:\nprint("\n=== TABELLE IN SQLITE ===")\nprint(pd.read_sql("SELECT name FROM sqlite_master WHERE type=\'table\';", conn))\n\nprint("\n=== CONTENUTO TABELLA SONDAGGI ===")\nprint(pd.read_sql("SELECT * FROM sondaggi;", conn))\n\nprint("\n=== CONTENUTO TABELLA ANALISI_CAMPIONI ===")\nprint(pd.read_sql("SELECT * FROM analisi_campioni;", conn))'

In [4]:
q_pulizia_1 = """
SELECT 
 -- fix ID_Sondaggio
CASE
    WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'SD[0-9]*' THEN SUBSTR(ID_Sondaggio, 1, 2) || '-' || SUBSTR(ID_Sondaggio, 3, 3) 
    ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

 -- fix Data_Sondaggio
CASE
    WHEN Data_Sondaggio LIKE '__/__/____' THEN 
        SUBSTR(Data_Sondaggio, 7, 4) || '-' ||
        SUBSTR(Data_Sondaggio, 4, 2) || '-' ||
        SUBSTR(Data_Sondaggio, 1, 2)
    WHEN Data_Sondaggio LIKE '%.%' THEN 
        REPLACE(Data_Sondaggio, '.', '-')
    ELSE Data_Sondaggio
END AS data_sondaggio,

 -- fix Zona
CASE
    WHEN LOWER(TRIM(Zona)) = 'zona a' THEN 'Zona A'
    WHEN TRIM(Zona) LIKE '%_%' THEN REPLACE(TRIM(Zona), '_', ' ')
    ELSE TRIM(Zona)
END AS zona,

 -- fix Profondita'_Totale_m
TRIM(CAST("Profondita'_Totale_m" AS REAL)) AS profondita_totale_m,

 -- fix Attrezzatura
LOWER(TRIM(Attrezzatura)) AS attrezzatura
FROM sondaggi;
"""

q_pulizia_2 = """
SELECT
 -- fix ID_Campione
CASE
    WHEN UPPER(TRIM(ID_Campione)) GLOB 'C[0-9]*' THEN 'CAMP-' || SUBSTR(ID_Campione, 2, 3) 
    ELSE UPPER(TRIM(ID_Campione))
END AS id_campione,

 -- fix ID_Sondaggio
CASE
    WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'SD[0-9]*' THEN SUBSTR(ID_Sondaggio, 1, 2) || '-' || SUBSTR(ID_Sondaggio, 3, 3) 
    ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

 -- fix Da_m
CAST(TRIM(Da_m) AS REAL) AS da_m,

 -- fix A_m
CAST(TRIM(A_m) AS REAL) AS a_m,

 -- fix Au_ppm
CASE
    WHEN Au_ppm = 'ND' THEN NULL 
    ELSE CAST(TRIM(Au_ppm) AS REAL) 
END AS Au_ppm_clean,

 -- fix Ag_ppm
CASE
    WHEN Ag_ppm = 'ND' THEN NULL 
    ELSE CAST(TRIM(Ag_ppm) AS REAL) 
END AS Ag_ppm_clean,

 -- fix Cu_ppm
CASE
    WHEN Cu_ppm = 'ND' THEN NULL 
    ELSE CAST(TRIM(Cu_ppm) AS REAL) 
END AS Cu_ppm_clean
FROM analisi_campioni;
"""

cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_1_temp AS " + q_pulizia_1)

print("=== DATI GREZZI SONDAGGI -PULITI!- ==")
df_verifica_1 = pd.read_sql_query("SELECT * FROM Campioni_puliti_1_temp", conn)
print(df_verifica_1)

print("\n\n")

cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_2_temp AS " + q_pulizia_2)

print("=== DATI GREZZI ANALISI CAMPIONI -PULITI!- ===")
df_verifica_2 = pd.read_sql_query("SELECT * FROM Campioni_puliti_2_temp", conn)
print(df_verifica_2)

=== DATI GREZZI SONDAGGI -PULITI!- ==
  id_sondaggio data_sondaggio    zona profondita_totale_m attrezzatura
0       SD-001     2024-01-10  Zona A                 0.0       rotary
1       SD-001     2024-02-10  Zona A                 0.0  percussione
2       SD-002     2024-03-15  Zona B                 0.0       rotary
3       SD-003     2024-04-01  Zona C                 0.0      diamond



=== DATI GREZZI ANALISI CAMPIONI -PULITI!- ===
  id_campione id_sondaggio  da_m   a_m  Au_ppm_clean  Ag_ppm_clean  \
0    CAMP-101       SD-001   0.0   1.0           0.5           2.1   
1    CAMP-101       SD-001  10.5  11.5           1.2           0.5   
2    CAMP-102       SD-002  15.0  16.0           NaN           1.5   
3    CAMP-103       SD-002  20.0  25.0           2.5           NaN   
4    CAMP-104       SD-001  25.0  30.0           0.0           0.0   

   Cu_ppm_clean  
0         120.0  
1         250.0  
2           NaN  
3          80.0  
4           0.0  
